In [ ]:
#@title 1. Install Dependencies
!pip install -q accelerate peft
!pip install -q git+https://github.com/huggingface/diffusers.git
!pip install -q gradio==5.6.0 gradio_client==1.4.3
!pip install -q -U transformers huggingface-hub
!pip install -q torch torchvision
!pip install -q bitsandbytes sentencepiece
!pip install -q pillow numpy scipy tqdm
!pip install -q spaces protobuf einops timm

# Clone CatVTON-FLUX
import os
if not os.path.exists('catvton-flux'):
    !git clone https://github.com/nftblackmagic/catvton-flux.git
os.chdir('catvton-flux')

print('\nDependencies installed!')

In [ ]:
#@title 2. Load CatVTON-FLUX Model (A100 - 40GB VRAM)
import torch
import gc

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name()} ({vram:.1f} GB VRAM)')

import psutil
ram_gb = psutil.virtual_memory().total / 1e9
print(f'System RAM: {ram_gb:.1f} GB')

# Load FLUX pipeline with CatVTON LoRA
from diffusers import FluxFillPipeline
from peft import LoraConfig
import huggingface_hub

print('\nLoading FLUX.1-Fill-dev base model...')
pipe = FluxFillPipeline.from_pretrained(
    'black-forest-labs/FLUX.1-Fill-dev',
    torch_dtype=torch.bfloat16,
).to(device)

print('Loading CatVTON-FLUX LoRA weights...')
pipe.load_lora_weights(
    'xiaozaa/catvton-flux-lora-alpha',
    weight_name='pytorch_lora_weights.safetensors'
)

gc.collect()
if device == 'cuda':
    torch.cuda.empty_cache()
    vram_used = torch.cuda.memory_allocated() / 1e9
    print(f'\nVRAM used: {vram_used:.1f} GB / {vram:.1f} GB')

print('\nCatVTON-FLUX loaded successfully!')

In [ ]:
#@title 3. Define Generation Function
from PIL import Image, ImageOps, ImageDraw
import io
import base64
import requests
import json
import numpy as np
import time

def download_image(url, max_dim=1024):
    """Download image from URL, resize, return PIL Image."""
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    img = Image.open(io.BytesIO(response.content)).convert('RGB')
    img.thumbnail((max_dim, max_dim), Image.LANCZOS)
    return img

def image_to_base64(img, format='JPEG'):
    """Convert PIL Image to base64 string."""
    buffer = io.BytesIO()
    img.save(buffer, format=format, quality=92)
    return base64.b64encode(buffer.getvalue()).decode('utf-8')

def create_garment_mask(person_img, cloth_type='upper_body'):
    """Create a simple garment region mask.
    For production, use DensePose/SCHP segmentation.
    This creates a rectangular mask over the clothing region."""
    w, h = person_img.size
    mask = Image.new('L', (w, h), 0)  # Black = keep
    draw = ImageDraw.Draw(mask)
    
    if cloth_type == 'upper_body':
        # Mask upper body region (roughly chest to waist)
        top = int(h * 0.15)   # Below face
        bottom = int(h * 0.55) # Above hips
        left = int(w * 0.15)
        right = int(w * 0.85)
    elif cloth_type == 'lower_body':
        top = int(h * 0.45)
        bottom = int(h * 0.85)
        left = int(w * 0.15)
        right = int(w * 0.85)
    elif cloth_type == 'dresses':
        top = int(h * 0.15)
        bottom = int(h * 0.85)
        left = int(w * 0.15)
        right = int(w * 0.85)
    else:
        top = int(h * 0.15)
        bottom = int(h * 0.55)
        left = int(w * 0.15)
        right = int(w * 0.85)
    
    draw.rectangle([left, top, right, bottom], fill=255)  # White = inpaint
    return mask

def prepare_catvton_input(person_img, garment_img, mask_img, target_w=768, target_h=1024):
    """Prepare the concatenated input for CatVTON-FLUX.
    CatVTON concatenates person (with mask) and garment side by side."""
    # Resize to target
    person_resized = person_img.resize((target_w, target_h), Image.LANCZOS)
    garment_resized = garment_img.resize((target_w, target_h), Image.LANCZOS)
    mask_resized = mask_img.resize((target_w, target_h), Image.LANCZOS)
    
    # Create masked person (inpaint region becomes gray)
    person_array = np.array(person_resized)
    mask_array = np.array(mask_resized)
    masked_person = person_array.copy()
    masked_person[mask_array > 127] = 128  # Gray out masked region
    masked_person_img = Image.fromarray(masked_person)
    
    # Concatenate: [garment | masked_person] side by side
    concat_w = target_w * 2
    concat_img = Image.new('RGB', (concat_w, target_h))
    concat_img.paste(garment_resized, (0, 0))
    concat_img.paste(masked_person_img, (target_w, 0))
    
    # Create concatenated mask (garment side = black/keep, person side = actual mask)
    concat_mask = Image.new('L', (concat_w, target_h), 0)
    concat_mask.paste(mask_resized, (target_w, 0))
    
    return concat_img, concat_mask, target_w, target_h

def generate_tryon(person_url, garment_url, description='upper_body',
                   num_steps=30, guidance_scale=30, seed=42):
    """Generate virtual try-on using CatVTON-FLUX."""
    start = time.time()
    
    try:
        # Download images
        print(f'Downloading person image...')
        person_img = download_image(person_url)
        print(f'Person: {person_img.size}')
        
        print(f'Downloading garment image...')
        garment_img = download_image(garment_url)
        print(f'Garment: {garment_img.size}')
        
        # Create mask for clothing region
        cloth_type = description if description in ['upper_body', 'lower_body', 'dresses'] else 'upper_body'
        print(f'Creating mask (type: {cloth_type})...')
        
        target_w, target_h = 768, 1024
        person_resized = person_img.resize((target_w, target_h), Image.LANCZOS)
        mask = create_garment_mask(person_resized, cloth_type)
        
        # Prepare CatVTON concatenated input
        concat_img, concat_mask, tw, th = prepare_catvton_input(
            person_img, garment_img, mask, target_w, target_h
        )
        
        # Convert mask to RGB for FLUX
        concat_mask_rgb = concat_mask.convert('RGB')
        
        # Run FLUX inpainting with CatVTON LoRA
        print(f'Running CatVTON-FLUX inference ({num_steps} steps)...')
        generator = torch.Generator(device=device).manual_seed(seed)
        
        result = pipe(
            image=concat_img,
            mask_image=concat_mask,
            prompt='',  # CatVTON is prompt-free
            num_inference_steps=num_steps,
            guidance_scale=guidance_scale,
            generator=generator,
            height=target_h,
            width=target_w * 2,
        ).images[0]
        
        # Extract the person side (right half) from the result
        result_person = result.crop((target_w, 0, target_w * 2, target_h))
        
        # Convert to base64
        result_b64 = image_to_base64(result_person)
        
        elapsed = time.time() - start
        print(f'Done! Took {elapsed:.1f}s')
        
        if device == 'cuda':
            vram_used = torch.cuda.memory_allocated() / 1e9
            print(f'VRAM used: {vram_used:.1f} GB')
        
        return json.dumps({
            'success': True,
            'image_base64': result_b64,
            'duration_seconds': round(elapsed, 1)
        })
        
    except Exception as e:
        elapsed = time.time() - start
        print(f'Error: {e}')
        import traceback
        traceback.print_exc()
        return json.dumps({
            'success': False,
            'error': str(e),
            'duration_seconds': round(elapsed, 1)
        })

print('Generation function defined!')

In [ ]:
#@title 4. Launch Gradio Server
import gradio as gr

def tryon_api(person_url, garment_url, description='upper_body'):
    """Gradio API endpoint for virtual try-on."""
    return generate_tryon(
        person_url=person_url,
        garment_url=garment_url,
        description=description,
        num_steps=30,
        guidance_scale=30,
        seed=42,
    )

with gr.Blocks(title='CatVTON-FLUX Server') as demo:
    gr.Markdown('# CatVTON-FLUX Virtual Try-On Server')
    gr.Markdown('SOTA virtual try-on powered by FLUX.1-Fill-dev + CatVTON LoRA')
    
    with gr.Row():
        with gr.Column():
            person_input = gr.Textbox(label='Person Image URL', placeholder='https://...')
            garment_input = gr.Textbox(label='Garment Image URL', placeholder='https://...')
            desc_input = gr.Textbox(label='Cloth Type', value='upper_body',
                                   placeholder='upper_body / lower_body / dresses')
            submit_btn = gr.Button('Generate Try-On', variant='primary')
        with gr.Column():
            output = gr.Textbox(label='Result JSON', lines=10)
    
    submit_btn.click(
        fn=tryon_api,
        inputs=[person_input, garment_input, desc_input],
        outputs=[output],
        api_name='tryon'
    )

print('Starting Gradio server...')
demo.launch(share=True, debug=True)